# Megatron-LM Sequence Parallelism & Activation Sharding

## 1. Executive Concept & Theoretical Foundations
In standard Tensor Parallelism (TP) for transformer models (Megatron-LM), matrix multiplications in Multi-Head Attention and Feed-Forward Networks are split along rows or columns across $N$ GPUs. However, intermediate layers—specifically LayerNorm, Dropout, and residual additions—are duplicated across all $N$ GPUs:

$$\text{Hidden States Shape} = [b, s, h]$$

Where:
- $b$ is micro-batch size
- $s$ is sequence length
- $h$ is hidden dimension

As sequence lengths grow from 2K to 32K or 128K tokens, the activation memory consumed by duplicated LayerNorm and Dropout layers accounts for more than **50% of total activation memory**.

### Sequence Parallelism (SP) Solution
Megatron Sequence Parallelism splits the sequence dimension $s$ across the tensor-parallel group size $N_{tp}$ during non-tensor-parallel regions:
- In Attention/MLP: Activations are full sequence $[b, s, h / N_{tp}]$
- In LayerNorm/Dropout: Activations are sharded by sequence $[b, s / N_{tp}, h]$

Transitioning between these two representations requires:
1. **Forward Pass**: `All-Gather` before Attention/MLP, followed by `Reduce-Scatter` after.
2. **Backward Pass**: Reverse communication pattern (`All-Gather` before backward gradient computation, `Reduce-Scatter` after).

Crucially, standard Tensor Parallelism already requires an `All-Reduce` after attention and MLP. Because:
$$\text{All-Reduce} = \text{Reduce-Scatter} + \text{All-Gather}$$
Sequence Parallelism replaces the single `All-Reduce` with an exact split of its two constituent phases, adding **zero additional communication bytes** while saving $(1 - 1/N_{tp})$ of LayerNorm activation memory!

In [ ]:
import numpy as np

class SimulatedSequenceParallelism:
    """
    Simulates Megatron-LM Sequence Parallelism across N virtual GPUs.
    Demonstrates sequence dimension sharding, All-Gather, and Reduce-Scatter primitives.
    """
    def __init__(self, world_size=4, hidden_dim=64):
        self.world_size = world_size
        self.hidden_dim = hidden_dim
        
    def shard_sequence(self, full_tensor):
        """Splits [batch, seq_len, hidden_dim] across GPUs along seq_len."""
        b, s, h = full_tensor.shape
        assert s % self.world_size == 0, "Sequence length must be divisible by world_size"
        chunk_len = s // self.world_size
        shards = [full_tensor[:, i*chunk_len:(i+1)*chunk_len, :] for i in range(self.world_size)]
        return shards
        
    def all_gather(self, shards):
        """Gathers [batch, seq_len/world_size, hidden_dim] into full [batch, seq_len, hidden_dim]."""
        return np.concatenate(shards, axis=1)
        
    def reduce_scatter(self, full_tensors):
        """
        Sums tensors across all ranks and scatters the result along the sequence dimension.
        Input: List of [batch, seq_len, hidden_dim] from each rank
        Output: List of [batch, seq_len/world_size, hidden_dim] for each rank
        """
        summed = np.sum(full_tensors, axis=0) # [b, s, h]
        return self.shard_sequence(summed)

# Verification Test
sim = SimulatedSequenceParallelism(world_size=4, hidden_dim=8)
batch, seq_len, hidden = 2, 16, 8
input_x = np.random.randn(batch, seq_len, hidden)

# Shard input for sequence-parallel LayerNorm
sharded_x = sim.shard_sequence(input_x)
print(f"Full Sequence Tensor Shape: {input_x.shape}")
print(f"Sharded Sequence Per GPU Shape: {sharded_x[0].shape}")
assert sharded_x[0].shape == (2, 4, 8)

# Simulate All-Gather before attention
gathered_x = sim.all_gather(sharded_x)
np.testing.assert_allclose(input_x, gathered_x)
print("All-Gather reconstruction matches original input bit-for-bit!")

## 2. Activation Memory Analytical Audit
Let us quantify the exact activation memory savings achieved by Megatron Sequence Parallelism across various context lengths.

In [ ]:
def activation_memory_breakdown(seq_len, hidden_dim=4096, num_layers=32, tp_size=8, bytes_per_elem=2):
    """
    Computes activation memory (in GiB) for standard TP vs TP + Sequence Parallelism.
    Assuming micro-batch size = 1.
    """
    # Standard TP: LayerNorm activations are duplicated across all TP ranks:
    # 2 * b * s * h * bytes (for input and output of norm/dropout per layer)
    standard_norm_mem = 2 * seq_len * hidden_dim * bytes_per_elem * num_layers
    
    # Sequence Parallel TP: LayerNorm activations are divided by tp_size
    sp_norm_mem = (2 * seq_len * hidden_dim * bytes_per_elem * num_layers) / tp_size
    
    # Core attention activations (queries, keys, values sharded by tp_size)
    attn_mem = (10 * seq_len * hidden_dim * bytes_per_elem * num_layers) / tp_size
    
    total_standard = (standard_norm_mem + attn_mem) / (1024**3)
    total_sp = (sp_norm_mem + attn_mem) / (1024**3)
    savings_pct = (1.0 - (total_sp / total_standard)) * 100
    
    return total_standard, total_sp, savings_pct

print("Activation Memory Comparison (TP Size = 8, Hidden = 4096, Layers = 32):")
print("-" * 65)
for s in [2048, 8192, 32768, 65536]:
    std, sp, sav = activation_memory_breakdown(s)
    print(f"Context {s:5d} tokens | Std TP: {std:6.2f} GiB | TP + SP: {sp:6.2f} GiB | Savings: {sav:5.1f}%")

## 3. High-Yield Interview Questions & Engineering Trade-offs

### Q1: Why does Megatron Sequence Parallelism require zero additional communication overhead over standard Tensor Parallelism?
**Answer:** Standard Tensor Parallelism already performs an `All-Reduce` at the end of both the self-attention and MLP blocks to sum partial matrix multiplication results across TP ranks. Mathematically, an `All-Reduce` consists of two successive operations: `Reduce-Scatter` followed by `All-Gather`. 
In Sequence Parallelism, instead of running an undivided `All-Reduce`, the system performs `Reduce-Scatter` immediately after the projection layer (leaving activations sharded along the sequence dimension for LayerNorm/Dropout), and then performs `All-Gather` right before the next attention or MLP projection layer. The total number of bytes transferred over the interconnect (NVLink) remains identical: $2 \times \frac{N-1}{N} \times \text{data size}$.

### Q2: What is the difference between Megatron Sequence Parallelism and Ring Attention?
**Answer:** 
- **Megatron Sequence Parallelism (SP)** is designed for intra-node tensor-parallel groups (typically 4 to 8 GPUs over high-bandwidth NVLink). It shards non-tensor-parallel regions (LayerNorm, Dropout) along the sequence dimension but still requires gathering full sequence representations during attention computation.
- **Ring Attention (and DeepSpeed Ulysses)** scales sequence lengths to millions of tokens across distributed nodes. It divides query, key, and value chunks across GPUs and rotates keys and values in a circular ring pattern while computing block-wise FlashAttention in parallel, eliminating the need to gather the full sequence at any single node.